# Mixed-All PoseT5 Cross-Source Evaluation
Evaluates the published mixed-all runtime artifact across `tsl51`, `thaisignvis`, and `youtube_sl25_thai` using a video-level split, then searches decoding settings on the same cross-source subset.

In [ ]:
import csv, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

def find_input_mount(slug: str) -> str:
    candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / 'datasets' / 'orbitorls' / slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return str(candidate)
    return ''

def materialize_manifest_dataset(dataset_root: str, working_slug: str) -> str:
    dataset_path = Path(dataset_root)
    manifest_path = dataset_path / 'manifest.csv'
    if not manifest_path.is_file():
        raise FileNotFoundError({'dataset_root': dataset_root, 'missing': 'manifest.csv'})

    def manifest_feature_probe(root: Path) -> dict:
        matched = 0
        total = 0
        samples = []
        with manifest_path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rel_path = str(row.get('npy_path', '')).replace('\\', '/')
                if not rel_path:
                    continue
                total += 1
                candidate = root / rel_path
                if candidate.is_file():
                    matched += 1
                    if matched >= 3:
                        break
                elif len(samples) < 3:
                    samples.append(rel_path)
        return {'matched': matched, 'total': total, 'missing_samples': samples}

    direct_features = dataset_path / 'features'
    direct_probe = manifest_feature_probe(dataset_path)
    if direct_probe['matched'] > 0:
        return str(dataset_path)

    nested_features = direct_features / 'features'
    if nested_features.is_dir() and any(nested_features.rglob('*.npy')):
        data_runtime = Path('/kaggle/working/datasets') / working_slug
        shutil.rmtree(data_runtime, ignore_errors=True)
        data_runtime.mkdir(parents=True, exist_ok=True)
        for name in ['manifest.csv', 'manifest_quality.json']:
            src = dataset_path / name
            if src.is_file():
                shutil.copy2(src, data_runtime / name)
        shutil.copytree(nested_features, data_runtime / 'features', dirs_exist_ok=True)
        repaired_probe = manifest_feature_probe(data_runtime)
        if repaired_probe['matched'] > 0:
            return str(data_runtime)

    archive_candidates = [dataset_path / 'features.zip']
    archive_candidates.extend(sorted(dataset_path.rglob('features.zip')))
    archive_path = next((path for path in archive_candidates if path.is_file()), None)
    if archive_path is None:
        raise FileNotFoundError({
            'dataset_root': dataset_root,
            'missing': 'features or features.zip',
            'visible': sorted(path.name for path in dataset_path.iterdir()),
        })

    data_runtime = Path('/kaggle/working/datasets') / working_slug
    shutil.rmtree(data_runtime, ignore_errors=True)
    data_runtime.mkdir(parents=True, exist_ok=True)
    for name in ['manifest.csv', 'manifest_quality.json']:
        src = dataset_path / name
        if src.is_file():
            shutil.copy2(src, data_runtime / name)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(data_runtime)

    materialized_features = data_runtime / 'features'
    extracted_probe = manifest_feature_probe(data_runtime)
    if not materialized_features.is_dir() or extracted_probe['matched'] == 0:
        raise FileNotFoundError({
            'dataset_root': dataset_root,
            'archive_path': str(archive_path),
            'materialized_root': str(data_runtime),
            'missing': 'materialized manifest feature files',
            'probe': extracted_probe,
        })
    return str(data_runtime)

def query_gpu_info() -> tuple[str, float, str]:
    result = subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,compute_cap,memory.total',
        '--format=csv,noheader',
    ], text=True).strip().splitlines()[0]
    name, capability, memory_total = [part.strip() for part in result.split(',', 2)]
    return name, float(capability), memory_total

def ensure_compatible_torch(gpu_capability: float) -> dict:
    selected = None
    if gpu_capability < 7.0:
        compatible_candidates = [
            ('2.2.0', '0.17.0', '2.2.0'),
            ('2.2.1', '0.17.1', '2.2.1'),
            ('2.2.2', '0.17.2', '2.2.2'),
            ('2.3.0', '0.18.0', '2.3.0'),
            ('2.3.1', '0.18.1', '2.3.1'),
            ('2.4.0', '0.19.0', '2.4.0'),
            ('2.4.1', '0.19.1', '2.4.1'),
            ('2.5.0', '0.20.0', '2.5.0'),
            ('2.5.1', '0.20.1', '2.5.1'),
        ]
        for torch_version, torchvision_version, torchaudio_version in compatible_candidates:
            subprocess.check_call([
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                f'torch=={torch_version}',
                f'torchvision=={torchvision_version}',
                f'torchaudio=={torchaudio_version}',
                '--index-url',
                'https://download.pytorch.org/whl/cu121',
            ])
            arch_probe = subprocess.check_output([
                sys.executable,
                '-c',
                'import json, torch; print(json.dumps(sorted(torch.cuda.get_arch_list())))',
            ], text=True).strip()
            print({'candidate_torch': torch_version, 'arch_probe': arch_probe})
            if 'sm_60' in arch_probe:
                selected = {
                    'torch': torch_version,
                    'torchvision': torchvision_version,
                    'torchaudio': torchaudio_version,
                    'wheel_index': 'cu121',
                }
                break
        if selected is None:
            raise RuntimeError('No available cu121 torch wheel in Kaggle exposed sm_60 support for P100.')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy<2'])
    return selected or {'torch': 'default', 'wheel_index': 'default'}

CODE_DATASET = find_input_mount('thai-sign-code')
DATA = find_input_mount('thai-sign-mixed-all-v6-archived')
EXPORT_DATASET = find_input_mount('thai-sign-mixed-all-output')
CODE = '/tmp/thai_sign_code_eval'
DATA_RUNTIME = materialize_manifest_dataset(DATA, 'thai-sign-mixed-all-v6-archived')
EXPORT_DIR = EXPORT_DATASET
OUT_DIR = '/kaggle/working/mixed_all_eval'

missing = [name for name, path in {
    'thai-sign-code': CODE_DATASET,
    'thai-sign-mixed-all-v6-archived': DATA,
    'thai-sign-mixed-all-output': EXPORT_DATASET,
}.items() if not path]
if missing:
    raise FileNotFoundError({'missing_inputs': missing, 'visible_inputs': sorted(p.name for p in INPUT_ROOT.iterdir())})

gpu_name, gpu_capability, gpu_memory = query_gpu_info()
selected_torch = ensure_compatible_torch(gpu_capability)

shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)
repo_bundle = os.path.join(CODE_DATASET, 'repo_bundle.zip')
if os.path.isfile(repo_bundle):
    with zipfile.ZipFile(repo_bundle) as archive:
        archive.extractall(CODE)
else:
    for name in ['config.py', 'requirements.txt', 'README.md']:
        src = os.path.join(CODE_DATASET, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(CODE, name))
    for name in ['src', 'scripts']:
        src_dir = os.path.join(CODE_DATASET, name)
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join(CODE, name), dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
for package in ['transformers>=4.40', 'sentencepiece>=0.2.0', 'sacrebleu>=2.4.0', 'portalocker>=2.0', 'pandas>=2.0']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print({
    'code_dataset': CODE_DATASET,
    'code_runtime': CODE,
    'data': DATA,
    'data_runtime': DATA_RUNTIME,
    'export_dataset': EXPORT_DATASET,
    'out_dir': OUT_DIR,
    'gpu_name': gpu_name,
    'gpu_capability': gpu_capability,
    'gpu_memory': gpu_memory,
    'selected_torch': selected_torch,
    'gpu_available': __import__('torch').cuda.is_available(),
})


In [ ]:
import os, subprocess, sys, torch

eval_script = os.path.join(CODE, 'scripts', 'evaluate_pose_t5_export.py')
search_script = os.path.join(CODE, 'scripts', 'search_pose_t5_decoding.py')
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([os.path.join(CODE, 'src'), CODE, env.get('PYTHONPATH', '')])
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

eval_cmd = [
    sys.executable,
    eval_script,
    '--export-dir', EXPORT_DIR,
    '--data-roots', DATA_RUNTIME,
    '--device', device,
    '--seed', '42',
    '--val-subset-size', '90',
    '--split-policy', 'video',
    '--required-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--manifest-quality-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--eval-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--report-data-roots', 'data/mixed_all_train_v6',
    '--report-json', os.path.join(OUT_DIR, 'mixed_all_cross_source_eval.json'),
    '--samples-json', os.path.join(OUT_DIR, 'mixed_all_cross_source_samples.json'),
]
print('Running:', ' '.join(eval_cmd))
subprocess.run(eval_cmd, check=True, env=env)

search_cmd = [
    sys.executable,
    search_script,
    '--export-dir', EXPORT_DIR,
    '--data-roots', DATA_RUNTIME,
    '--device', device,
    '--seed', '42',
    '--val-subset-size', '60',
    '--split-policy', 'video',
    '--required-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--manifest-quality-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--eval-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--report-data-roots', 'data/mixed_all_train_v6',
    '--report-json', os.path.join(OUT_DIR, 'mixed_all_cross_source_decoding_search.json'),
    '--best-eval-json', os.path.join(OUT_DIR, 'mixed_all_cross_source_decoding_best_eval.json'),
    '--best-samples-json', os.path.join(OUT_DIR, 'mixed_all_cross_source_decoding_best_samples.json'),
    '--max-trials', '12',
]
print('Running:', ' '.join(search_cmd))
try:
    subprocess.run(search_cmd, check=True, env=env)
except subprocess.CalledProcessError:
    if device != 'cuda':
        raise
    retry_cmd = list(search_cmd)
    retry_cmd[retry_cmd.index('--device') + 1] = 'cpu'
    retry_cmd[retry_cmd.index('--val-subset-size') + 1] = '30'
    retry_cmd[retry_cmd.index('--max-trials') + 1] = '6'
    print('Retrying decoding search on CPU:', ' '.join(retry_cmd))
    subprocess.run(retry_cmd, check=True, env=env)


In [ ]:
import json, os
from pathlib import Path

print(sorted(os.listdir(OUT_DIR)))
for name in [
    'mixed_all_cross_source_eval.json',
    'mixed_all_cross_source_decoding_best_eval.json',
]:
    path = Path(OUT_DIR) / name
    if path.is_file():
        print(name)
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:4000])
